# 01 Setup and Processing

prepares the **FakeHealth + healthfact** dataset for model training, It reads the raw combined CSV, performs light cleaning, keeps only binary-labeled rows, and writes the processed files back to the project folder.


In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd


In [2]:
PROJECT_ROOT = Path(r'C:\Users\ribam\Desktop\Reseach\Dataset')
DATA_ROOT = PROJECT_ROOT / 'dataset'
RAW_DIR = DATA_ROOT / 'raw'
PROCESSED_DIR = DATA_ROOT / 'processed'
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'

RAW_PATH = RAW_DIR / 'fakehealth_healthfact_combined.csv'
PROCESSED_PATH = PROCESSED_DIR / 'fakehealth_healthfact_binary_clean.csv'
SUMMARY_PATH = PROCESSED_DIR / 'fakehealth_healthfact_processing_summary.json'

RAW_PATH, PROCESSED_PATH, SUMMARY_PATH


(WindowsPath('C:/Users/ribam/Desktop/Reseach/Dataset/dataset/raw/fakehealth_healthfact_combined.csv'),
 WindowsPath('C:/Users/ribam/Desktop/Reseach/Dataset/dataset/processed/fakehealth_healthfact_binary_clean.csv'),
 WindowsPath('C:/Users/ribam/Desktop/Reseach/Dataset/dataset/processed/fakehealth_healthfact_processing_summary.json'))

In [3]:
df = pd.read_csv(RAW_PATH)
print('Raw shape:', df.shape)
df.head(3)


Raw shape: (14422, 11)


,dataset,split,record_id,text_kind,title,text,url,source,publish_date,label_original,label_binary
0,fakehealth,release,news_reviews_00000,article_text,Tiny implantable device short-circuits hunger ...,"MADISON, Wis. -- More than 700 million adults ...",https://web.archive.org/web/20181218015531/htt...,https://web.archive.org,1.546060e+09,2,0.0
1,fakehealth,release,news_reviews_00001,article_text,Scientists report CRISPR restores effectivenes...,"Wilmington, DE, December 17, 2018 - The CRISPR...",https://web.archive.org/web/20181217203805/htt...,https://web.archive.org,1.546060e+09,3,1.0
2,fakehealth,release,news_reviews_00002,article_text,Probiotics could help millions of patients suf...,About 3 million people in the US are diagnosed...,https://web.archive.org/web/20181213085845/htt...,https://web.archive.org,1.546060e+09,1,0.0


In [4]:
audit = {
    'dataset_counts': df['dataset'].value_counts(dropna=False).to_dict(),
    'split_counts': df['split'].value_counts(dropna=False).to_dict(),
    'text_kind_counts': df['text_kind'].value_counts(dropna=False).to_dict(),
    'label_original_counts': df['label_original'].fillna('').astype(str).value_counts(dropna=False).to_dict(),
    'label_binary_counts': df['label_binary'].fillna('').astype(str).value_counts(dropna=False).to_dict(),
    'missing_text_rows': int(df['text'].isna().sum()),
}
audit


{'dataset_counts': {'healthfact': 12266, 'fakehealth': 2156},
 'split_counts': {'train': 9814,
  'story': 1564,
  'test': 1235,
  'dev': 1217,
  'release': 592},
 'text_kind_counts': {'claim': 12266, 'article_text': 2156},
 'label_original_counts': {'TRUE': 6306,
  'FALSE': 3769,
  'mixture': 1799,
  '3': 707,
  '2': 509,
  '4': 498,
  'unproven': 377,
  '5': 228,
  '1': 183,
  '0': 31,
  '': 15},
 'label_binary_counts': {'1.0': 7542, '0.0': 4689, '': 2191},
 'missing_text_rows': 0}

## Cleaning Rules

The processing below keeps the setup simple and reproducible:

- strip repeated whitespace from `title` and `text`
- normalize `label_binary` to only `0` or `1`
- drop rows without a binary label
- keep one row per `dataset + record_id`
- add a numeric `label` column for training
- add basic text-length features for quick EDA


In [5]:
def normalize_space(value):
    if pd.isna(value):
        return ''
    value = str(value)
    return re.sub(r'\s+', ' ', value).strip()

def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan

    text_value = str(value).strip()
    if text_value in {'0', '1'}:
        return text_value

    try:
        numeric_value = float(text_value)
    except ValueError:
        return np.nan

    if numeric_value == 0.0:
        return '0'
    if numeric_value == 1.0:
        return '1'
    return np.nan


In [6]:
df_processed = df.copy()
df_processed['title'] = df_processed['title'].apply(normalize_space)
df_processed['text'] = df_processed['text'].apply(normalize_space)
df_processed['label_binary'] = df_processed['label_binary'].apply(normalize_binary_label)

df_processed = df_processed[df_processed['text'] != ''].copy()
df_processed = df_processed[df_processed['label_binary'].isin(['0', '1'])].copy()
df_processed = df_processed.drop_duplicates(subset=['dataset', 'record_id']).copy()

df_processed['label'] = df_processed['label_binary'].astype(int)
df_processed['text_length_chars'] = df_processed['text'].str.len()
df_processed['text_length_words'] = df_processed['text'].str.split().str.len()

column_order = [
    'dataset', 'split', 'record_id', 'text_kind', 'title', 'text',
    'url', 'source', 'publish_date', 'label_original', 'label_binary',
    'label', 'text_length_chars', 'text_length_words'
]
df_processed = df_processed[column_order]

print('Processed shape:', df_processed.shape)
df_processed.head(3)


Processed shape: (12231, 14)


,dataset,split,record_id,text_kind,title,text,url,source,publish_date,label_original,label_binary,label,text_length_chars,text_length_words
0,fakehealth,release,news_reviews_00000,article_text,Tiny implantable device short-circuits hunger ...,"MADISON, Wis. -- More than 700 million adults ...",https://web.archive.org/web/20181218015531/htt...,https://web.archive.org,1.546060e+09,2,0,0,3615,553
1,fakehealth,release,news_reviews_00001,article_text,Scientists report CRISPR restores effectivenes...,"Wilmington, DE, December 17, 2018 - The CRISPR...",https://web.archive.org/web/20181217203805/htt...,https://web.archive.org,1.546060e+09,3,1,1,7022,1136
2,fakehealth,release,news_reviews_00002,article_text,Probiotics could help millions of patients suf...,About 3 million people in the US are diagnosed...,https://web.archive.org/web/20181213085845/htt...,https://web.archive.org,1.546060e+09,1,0,0,2967,453


In [7]:
df_processed.groupby(['dataset', 'label_binary']).size().unstack(fill_value=0)


label_binary,0,1
dataset,,
fakehealth,920,1236
healthfact,3769,6306


In [8]:
summary = {
    'raw_rows': int(len(df)),
    'processed_rows': int(len(df_processed)),
    'dropped_missing_binary': int(df['label_binary'].fillna('').astype(str).eq('').sum()),
    'dataset_counts_processed': df_processed['dataset'].value_counts().to_dict(),
    'label_counts_processed': df_processed['label_binary'].value_counts().to_dict(),
    'split_counts_processed': df_processed['split'].value_counts(dropna=False).to_dict(),
}

df_processed.to_csv(PROCESSED_PATH, index=False)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Saved processed CSV to:', PROCESSED_PATH)
print('Saved summary JSON to:', SUMMARY_PATH)
summary


Saved processed CSV to: C:\Users\ribam\Desktop\Reseach\Dataset\dataset\processed\fakehealth_healthfact_binary_clean.csv
Saved summary JSON to: C:\Users\ribam\Desktop\Reseach\Dataset\dataset\processed\fakehealth_healthfact_processing_summary.json


{'raw_rows': 14422,
 'processed_rows': 12231,
 'dropped_missing_binary': 2191,
 'dataset_counts_processed': {'healthfact': 10075, 'fakehealth': 2156},
 'label_counts_processed': {'1': 7542, '0': 4689},
 'split_counts_processed': {'train': 8079,
  'story': 1564,
  'dev': 1009,
  'test': 987,
  'release': 592}}

After this, the next notebook is **EDA(Exploratory Data Analysis) and label analysis**, followed by **model preparation / train-test strategy**.
